# Train-Serve Skew Analysis

Side-by-side comparison of Feast offline features (`get_historical_features`) vs online features (`get_online_features`) materialized in Redis.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from feature_store.feast_client import get_feature_store

FEATURE_REPO_PATH = PROJECT_ROOT / "feature_store" / "feature_repo"
META_PATH = PROJECT_ROOT / "data" / "training_meta.json"
PROCESSED_FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "vendor_features.parquet"

FEATURE_REFS = [
    "vendor_stats:trip_count_last_7d",
    "vendor_stats:avg_fare_last_7d",
    "vendor_stats:avg_trip_distance_last_7d",
    "vendor_stats:avg_passenger_count",
    "vendor_stats:peak_hour_ratio",
]

with META_PATH.open("r", encoding="utf-8") as handle:
    training_meta = json.load(handle)

feature_columns = training_meta["feature_columns"]
training_timestamp = pd.Timestamp(training_meta["training_timestamp"])

table = pq.read_table(PROCESSED_FEATURES_PATH, columns=["vendor_id"])
vendor_ids = (
    table.to_pandas()["vendor_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

store = get_feature_store(FEATURE_REPO_PATH)
entity_df = pd.DataFrame({"vendor_id": vendor_ids, "event_timestamp": training_timestamp})
print(f"Loaded {len(vendor_ids)} vendors for analysis at {training_timestamp}")

## 1. Offline Features

In [ ]:
offline_df = store.get_historical_features(
    entity_df=entity_df,
    features=FEATURE_REFS,
).to_df()

offline_df = offline_df.dropna(subset=feature_columns).copy()
for column in feature_columns:
    offline_df[column] = offline_df[column].astype(np.float32)

offline_df = offline_df.sort_values("vendor_id").reset_index(drop=True)
offline_df.head()

## 2. Online Features

In [ ]:
entity_rows = [{"vendor_id": vendor_id} for vendor_id in vendor_ids]
online_df = store.get_online_features(
    features=FEATURE_REFS,
    entity_rows=entity_rows,
).to_df()

for column in feature_columns:
    online_df[column] = online_df[column].astype(np.float32)

online_df = online_df.sort_values("vendor_id").reset_index(drop=True)
online_df.head()

## 3. Side-by-Side Comparison

In [ ]:
comparison = offline_df[["vendor_id", *feature_columns]].merge(
    online_df[["vendor_id", *feature_columns]],
    on="vendor_id",
    suffixes=("_offline", "_online"),
)

side_by_side_rows = []
for _, row in comparison.iterrows():
    for feature in feature_columns:
        side_by_side_rows.append(
            {
                "vendor_id": row["vendor_id"],
                "feature": feature,
                "offline": round(float(row[f"{feature}_offline"]), 4),
                "online": round(float(row[f"{feature}_online"]), 4),
            }
        )

side_by_side = pd.DataFrame(side_by_side_rows)
side_by_side.head(10)

## 4. Delta Table (Max Absolute Difference)

In [ ]:
delta_rows = []
for feature in feature_columns:
    offline_values = comparison[f"{feature}_offline"].astype(float)
    online_values = comparison[f"{feature}_online"].astype(float)
    max_abs_delta = float(np.max(np.abs(offline_values - online_values)))
    delta_rows.append({"feature": feature, "max_abs_delta": max_abs_delta})

delta_table = pd.DataFrame(delta_rows)
delta_table

## 5. Dtype Comparison

In [ ]:
dtype_rows = []
for feature in feature_columns:
    dtype_rows.append(
        {
            "feature": feature,
            "offline_dtype": str(offline_df[feature].dtype),
            "online_dtype": str(online_df[feature].dtype),
            "training_dtype": training_meta["feature_dtypes"][feature],
        }
    )

pd.DataFrame(dtype_rows)

## 6. SHA256 Hash Comparison

In [ ]:
def compute_hash(df: pd.DataFrame) -> str:
    sorted_df = df.sort_values("vendor_id").reset_index(drop=True)
    matrix = sorted_df[feature_columns].round(4).to_numpy(dtype=np.float64)
    return hashlib.sha256(np.round(matrix, 4).tobytes()).hexdigest()

offline_hash = compute_hash(offline_df)
online_hash = compute_hash(online_df)
expected_hash = training_meta["feature_hash"]

hash_match = offline_hash == online_hash == expected_hash
color = "green" if hash_match else "red"
label = "MATCH" if hash_match else "MISMATCH"

display(
    HTML(
        f"<h3 style='color:{color};'>SHA256 Hash Comparison: {label}</h3>"
        f"<p>offline={offline_hash}<br/>online={online_hash}<br/>training_meta={expected_hash}</p>"
    )
)

## 7. Feature Distribution Bar Chart

In [ ]:
fig, axes = plt.subplots(len(feature_columns), 1, figsize=(10, 16))

for axis, feature in zip(axes, feature_columns):
    offline_values = offline_df[feature].astype(float)
    online_values = online_df[feature].astype(float)
    bins = np.linspace(
        min(offline_values.min(), online_values.min()),
        max(offline_values.max(), online_values.max()),
        20,
    )
    axis.hist(offline_values, bins=bins, alpha=0.6, label="offline", color="tab:blue")
    axis.hist(online_values, bins=bins, alpha=0.6, label="online", color="tab:orange")
    axis.set_title(feature)
    axis.legend()

plt.tight_layout()
plt.show()